In [6]:
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
import glob

# Define class colors and mapping
CLASS_COLORS = {
    1: (166, 202, 240),   # airplane
    2: (128, 128, 0),     # bare soil
    3: (0, 0, 128),       # buildings
    4: (255, 0, 0),       # cars
    5: (0, 128, 0),       # chaparral
    6: (128, 0, 0),       # court
    7: (255, 233, 233),   # dock
    8: (160, 160, 164),   # field
    9: (0, 128, 128),     # grass
    10: (90, 87, 255),    # mobile home
    11: (255, 255, 0),    # pavement
    12: (255, 192, 0),    # sand
    13: (0, 0, 255),      # sea
    14: (255, 0, 92),     # ship
    15: (128, 0, 128),    # tanks
    16: (0, 255, 0),      # trees
    17: (0, 255, 255),    # water
}

BACKGROUND_COLOR = (100, 100, 100)
NUM_CLASSES = len(CLASS_COLORS) + 1

# Create a mapping from RGB to class index
COLOR_TO_CLASS = {}
for class_idx, color in CLASS_COLORS.items():
    COLOR_TO_CLASS[color] = class_idx

# Model architecture (same as your training)
class ConstrainedSegmentationModel(torch.nn.Module):
    def __init__(self, num_classes, backbone='resnet34'):
        super(ConstrainedSegmentationModel, self).__init__()
        self.num_classes = num_classes
        
        if backbone == 'resnet34':
            self.encoder = torch.hub.load('pytorch/vision:v0.10.0', 'resnet34', pretrained=True)
            self.encoder = torch.nn.Sequential(*list(self.encoder.children())[:-2])
            encoder_channels = 512
        
        self.point_conv = torch.nn.Sequential(
            torch.nn.Conv2d(num_classes, 64, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(inplace=True),
            torch.nn.Conv2d(64, 128, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(128),
            torch.nn.ReLU(inplace=True)
        )
        
        self.boundary_conv = torch.nn.Sequential(
            torch.nn.Conv2d(1, 64, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(inplace=True),
            torch.nn.Conv2d(64, 128, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(128),
            torch.nn.ReLU(inplace=True)
        )
        
        self.class_presence = torch.nn.Sequential(
            torch.nn.AdaptiveAvgPool2d(1),
            torch.nn.Conv2d(num_classes, 64, kernel_size=1),
            torch.nn.ReLU(inplace=True),
            torch.nn.Conv2d(64, num_classes, kernel_size=1),
            torch.nn.Sigmoid()
        )
        
        self.decoder = torch.nn.Sequential(
            torch.nn.Conv2d(encoder_channels + 128 + 128, 512, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(512),
            torch.nn.ReLU(inplace=True),
            torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            torch.nn.Conv2d(512, 256, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(256),
            torch.nn.ReLU(inplace=True),
            torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            torch.nn.Conv2d(256, 128, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(128),
            torch.nn.ReLU(inplace=True),
            torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            torch.nn.Conv2d(128, 64, kernel_size=3, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(inplace=True),
            torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            torch.nn.Conv2d(64, num_classes, kernel_size=1)
        )
        
    def forward(self, x, point_heatmap, boundary_heatmap):
        image_features = self.encoder(x)
        point_features = self.point_conv(point_heatmap)
        point_features = F.interpolate(point_features, size=image_features.shape[2:], mode='bilinear', align_corners=True)
        boundary_features = self.boundary_conv(boundary_heatmap)
        boundary_features = F.interpolate(boundary_features, size=image_features.shape[2:], mode='bilinear', align_corners=True)
        class_presence_mask = self.class_presence(point_heatmap)
        combined = torch.cat([image_features, point_features, boundary_features], dim=1)
        output = self.decoder(combined)
        output = F.interpolate(output, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=True)
        output = output * class_presence_mask
        return output

def load_trained_model(model_path, device):
    """Load the trained model from saved weights"""
    model = ConstrainedSegmentationModel(NUM_CLASSES, backbone='resnet34')
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model

def predict_dense_annotation(model, image_path, point_mask_path, device):
    """Predict dense annotation for a single image - EXACTLY AS YOUR WORKING VERSION"""
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    original_size = image.size
    img_tensor = torch.from_numpy(np.array(image).transpose(2, 0, 1).astype(np.float32) / 255.0)
    
    # Load and preprocess point mask
    point_mask_img = Image.open(point_mask_path).convert('RGB')
    point_data = np.array(point_mask_img)
    point_heatmap = np.zeros((NUM_CLASSES, original_size[1], original_size[0]), dtype=np.float32)
    
    # Track which classes are present
    present_classes = set()
    
    # Convert RGB point annotations to heatmap
    for y in range(point_data.shape[0]):
        for x in range(point_data.shape[1]):
            pixel = tuple(point_data[y, x])
            if pixel in COLOR_TO_CLASS:
                class_idx = COLOR_TO_CLASS[pixel]
                point_heatmap[class_idx, y, x] = 1.0
                present_classes.add(class_idx)
    
    point_heatmap_tensor = torch.from_numpy(point_heatmap)
    
    # Create dummy boundary heatmap (EXACTLY AS YOUR WORKING VERSION)
    boundary_heatmap = torch.zeros((original_size[1], original_size[0]))
    boundary_heatmap = boundary_heatmap.unsqueeze(0)
    
    # Add batch dimension
    img_tensor = img_tensor.unsqueeze(0).to(device)
    point_heatmap_tensor = point_heatmap_tensor.unsqueeze(0).to(device)
    boundary_heatmap = boundary_heatmap.unsqueeze(0).to(device)
    
    # Predict (EXACTLY AS YOUR WORKING VERSION)
    with torch.no_grad():
        output = model(img_tensor, point_heatmap_tensor, boundary_heatmap)
        probs = torch.softmax(output, dim=1)
        
        # Enforce class constraint
        for class_idx in range(1, NUM_CLASSES):
            if class_idx not in present_classes:
                probs[:, class_idx, :, :] = 0
        
        probs = probs / (probs.sum(dim=1, keepdim=True) + 1e-10)
        preds = torch.argmax(probs, dim=1)
    
    # Convert to numpy
    prediction_np = preds.squeeze(0).cpu().numpy()
    
    # Convert prediction to colored mask
    colored_mask = np.zeros((prediction_np.shape[0], prediction_np.shape[1], 3), dtype=np.uint8)
    for class_idx, color in CLASS_COLORS.items():
        colored_mask[prediction_np == class_idx] = color
    colored_mask[prediction_np == 0] = [0, 0, 0]  # Black background
    
    return prediction_np, colored_mask, image, present_classes

def display_only_predictions(model, image_folder, point_masks_folder, device, num_samples=5, save_dir=None):
    """
    Display only the dense predictions and save them if requested
    """
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
    
    # Get list of images
    image_files = sorted(glob.glob(os.path.join(image_folder, "*.png")))
    sample_ids = [os.path.splitext(os.path.basename(f))[0] for f in image_files]
    
    # Select samples to visualize
    selected_ids = sample_ids[:min(num_samples, len(sample_ids))]
    
    # Create figure with only predictions
    fig, axes = plt.subplots(1, len(selected_ids), figsize=(5 * len(selected_ids), 5))
    
    # If only one sample, reshape axes
    if len(selected_ids) == 1:
        axes = [axes]
    
    for i, sample_id in enumerate(selected_ids):
        # Paths
        image_path = os.path.join(image_folder, f"{sample_id}.png")
        point_mask_path = os.path.join(point_masks_folder, f"{sample_id}.png")
        
        # Get model prediction
        prediction, colored_prediction, _, present_classes = predict_dense_annotation(
            model, image_path, point_mask_path, device
        )
        
        # Display prediction
        axes[i].imshow(colored_prediction)
        predicted_classes = set(np.unique(prediction))
        predicted_classes.discard(0)
        axes[i].set_title(f'{sample_id}\nClasses: {sorted(predicted_classes)}', 
                          fontsize=12, fontweight='bold', pad=15)
        axes[i].axis('off')
        
        # Save prediction if requested
        if save_dir:
            pred_path = os.path.join(save_dir, f"{sample_id}.png")
            Image.fromarray(colored_prediction).save(pred_path)
            print(f"Saved prediction to {pred_path}")
    
    plt.tight_layout()
    plt.show()

# Main execution
if __name__ == "__main__":
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Your folder paths - UPDATE THESE WITH YOUR ACTUAL PATHS
    POINT_MASKS_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_masks"
    IMAGE_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images"
    MODEL_PATH = 'final_constrained_model_v3.pth'
    
    # Load your trained model
    print("Loading trained model...")
    model = load_trained_model(MODEL_PATH, device)
    print("Model loaded successfully!")
    
    # Display only predictions
    print("Displaying dense predictions...")
    display_only_predictions(
        model=model,
        image_folder=IMAGE_FOLDER,
        point_masks_folder=POINT_MASKS_FOLDER,
        device=device,
        num_samples=630,
        save_dir="dense_predictions"
    )
    print("Dense prediction visualization complete!")

Using device: cuda
Loading trained model...


Using cache found in /home/iiitdmk-param/.cache/torch/hub/pytorch_vision_v0.10.0
/tmp/ipykernel_1002271/2359522861.py:111: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mode

Model loaded successfully!
Displaying dense predictions...
Saved prediction to dense_predictions/1.png
Saved prediction to dense_predictions/10.png
Saved prediction to dense_predictions/100.png
Saved prediction to dense_predictions/101.png
Saved prediction to dense_predictions/102.png
Saved prediction to dense_predictions/103.png
Saved prediction to dense_predictions/104.png
Saved prediction to dense_predictions/105.png
Saved prediction to dense_predictions/106.png
Saved prediction to dense_predictions/107.png
Saved prediction to dense_predictions/108.png
Saved prediction to dense_predictions/109.png
Saved prediction to dense_predictions/11.png
Saved prediction to dense_predictions/110.png
Saved prediction to dense_predictions/111.png
Saved prediction to dense_predictions/112.png
Saved prediction to dense_predictions/113.png
Saved prediction to dense_predictions/114.png
Saved prediction to dense_predictions/115.png
Saved prediction to dense_predictions/116.png
Saved prediction to dense

ValueError: Image size of 315000x500 pixels is too large. It must be less than 2^16 in each direction.

Error in callback <function _draw_all_if_interactive at 0x7f73420b2c10> (for post_execute):


ValueError: Image size of 315000x500 pixels is too large. It must be less than 2^16 in each direction.

ValueError: Image size of 315000x500 pixels is too large. It must be less than 2^16 in each direction.

<Figure size 315000x500 with 630 Axes>

In [8]:
import os
import glob
POINT_MASKS_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_masks"
IMAGE_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images"
ORIGINAL_MASKS_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"
SAM_EDGES_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_predictions_boundary"
DENSE_PREDICTIONS_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/dense_predictions"

# Get sorted list of image paths from each folder
def get_sorted_image_paths(folder_path, extensions=['.jpg', '.jpeg', '.png', '.tif', '.tiff']):
    """
    Get sorted list of image file paths from a folder
    """
    image_paths = []
    for ext in extensions:
        pattern = os.path.join(folder_path, f"*{ext}")
        image_paths.extend(glob.glob(pattern))
    
    # Sort the paths for consistency
    image_paths.sort()
    return image_paths

# Generate the 5 lists
list1 = get_sorted_image_paths(POINT_MASKS_FOLDER)
list2 = get_sorted_image_paths(IMAGE_FOLDER)
list3 = get_sorted_image_paths(ORIGINAL_MASKS_FOLDER)
list4 = get_sorted_image_paths(SAM_EDGES_FOLDER)
list5 = get_sorted_image_paths(DENSE_PREDICTIONS_FOLDER)

# Print the number of images in each list
print(f"List1 (Point Masks): {len(list1)} images")
print(f"List2 (Images): {len(list2)} images")
print(f"List3 (Original Masks): {len(list3)} images")
print(f"List4 (SAM Edges): {len(list4)} images")
print(f"List5 (Dense Predictions): {len(list5)} images")

# Optional: Print first few paths from each list to verify
print("\nFirst 3 paths from each list:")
print("List1:", list1[:3])
print("List2:", list2[:3])
print("List3:", list3[:3])
print("List4:", list4[:3])
print("List5:", list5[:3])

List1 (Point Masks): 630 images
List2 (Images): 630 images
List3 (Original Masks): 630 images
List4 (SAM Edges): 630 images
List5 (Dense Predictions): 630 images

First 3 paths from each list:
List1: ['/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_masks/1.png', '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_masks/10.png', '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_masks/100.png']
List2: ['/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images/1.png', '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images/10.png', '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images/100.png']
List3: ['/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1/1.png', '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1/10.png', '/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1/100.png']
List4: ['/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM

In [9]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import numpy as np

# Create a new directory for saving the comparison plots
output_dir = "comparison_plots"
os.makedirs(output_dir, exist_ok=True)



# Combine all lists
all_lists = [list1, list2, list3, list4, list5]
list_names = ["List 1", "List 2", "List 3", "List 4", "List 5"]

# Find the minimum length among all lists to avoid index errors
min_length = min(len(lst) for lst in all_lists)

def create_comparison_plot(index):
    """
    Create a comparison plot for images at the given index across all lists
    """
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    fig.suptitle(f'Image Comparison - Index {index}', fontsize=16, fontweight='bold')    
    for i, (ax, lst, name) in enumerate(zip(axes, all_lists, list_names)):
        try:
            if index < len(lst):
                img_path = lst[index]
                
                # Load and display image
                img = mpimg.imread(img_path)
                ax.imshow(img)
                ax.set_title(f'{name}\n{os.path.basename(img_path)}', fontsize=10)
            else:
                # Create blank image if index is out of range
                ax.imshow(np.ones((100, 100, 3)))
                ax.set_title(f'{name}\nNo image', fontsize=10)
                
        except Exception as e:
            # Handle any errors in loading images
            ax.imshow(np.ones((100, 100, 3)))
            ax.set_title(f'{name}\nError loading image\n{str(e)[:30]}...', fontsize=8)
        
        ax.axis('off')
    
    plt.tight_layout()
    return fig

# Create comparison plots for each index
for index in range(min_length):
    fig = create_comparison_plot(index)
    
    # Save the plot
    output_path = os.path.join(output_dir, f'comparison_index_{index:03d}.png')
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    
    print(f"Saved comparison plot for index {index}")

print(f"All comparison plots saved in '{output_dir}' folder!")

# Optional: Create a summary plot showing first few comparisons
def create_summary_plot(num_comparisons=3):
    """Create a summary plot showing multiple comparisons"""
    num_rows = min(num_comparisons, min_length)
    
    fig, axes = plt.subplots(num_rows, 5, figsize=(20, 4 * num_rows))
    
    if num_rows == 1:
        axes = axes.reshape(1, -1)
    
    for row in range(num_rows):
        for col in range(5):
            ax = axes[row, col]
            try:
                if row < len(all_lists[col]):
                    img_path = all_lists[col][row]
                    img = mpimg.imread(img_path)
                    ax.imshow(img)
                    
                    if row == 0:  # Only show column titles on first row
                        ax.set_title(f'{list_names[col]}', fontsize=12)
                else:
                    ax.imshow(np.ones((100, 100, 3)))
                    
            except Exception as e:
                ax.imshow(np.ones((100, 100, 3)))
                ax.set_title('Error', fontsize=8)
            
            ax.axis('off')
            ax.set_ylabel(f'Index {row}', fontsize=10, rotation=0, ha='right', va='center')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'summary_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()

# Create summary plot
create_summary_plot()
print("Summary plot created!")



 

Saved comparison plot for index 0
Saved comparison plot for index 1
Saved comparison plot for index 2
Saved comparison plot for index 3
Saved comparison plot for index 4
Saved comparison plot for index 5
Saved comparison plot for index 6
Saved comparison plot for index 7
Saved comparison plot for index 8
Saved comparison plot for index 9
Saved comparison plot for index 10
Saved comparison plot for index 11
Saved comparison plot for index 12
Saved comparison plot for index 13
Saved comparison plot for index 14
Saved comparison plot for index 15
Saved comparison plot for index 16
Saved comparison plot for index 17
Saved comparison plot for index 18
Saved comparison plot for index 19
Saved comparison plot for index 20
Saved comparison plot for index 21
Saved comparison plot for index 22
Saved comparison plot for index 23
Saved comparison plot for index 24
Saved comparison plot for index 25
Saved comparison plot for index 26
Saved comparison plot for index 27
Saved comparison plot for inde